# 360 City Analyser en Colab

Runtime > Change runtime type > **T4 GPU** (o superior). Las celdas estan pensadas para ejecutarse en orden.
Nada de esto se ha ejecutado en Colab todavia: si una celda falla, pega el error.

## 1. Drive, repositorio y entorno

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/city-analyser'      # data, models y caches persisten aqui
os.makedirs(WORK + '/data', exist_ok=True); os.makedirs(WORK + '/models', exist_ok=True)

In [ ]:
# Repositorio privado: guarda un token de GitHub (solo lectura) en Colab > Secrets con el nombre GITHUB_TOKEN.
# Si es publico, basta con la URL sin token. Alternativa: subir un zip y descomprimirlo en /content/repo.
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/repo && git clone https://{token}@github.com/JSempereH/360_city_analyser.git /content/repo

In [ ]:
%cd /content/repo
!pip -q install uv
!uv sync --extra cu126
# data/ y models/ viven en Drive; el repo los ve como siempre
!rm -rf data models && ln -s {WORK}/data data && ln -s {WORK}/models models

In [ ]:
import os
os.environ['BUILDING_ANALYSIS_MODEL'] = 'high'                       # en T4 de 16 GB; 'balanced' si falta memoria
os.environ['BUILDING_ANALYSIS_STAGE_CACHE_DIR'] = WORK + '/stage-cache'
os.environ['BUILDING_ANALYSIS_OSM_CACHE_DIR'] = WORK + '/osm-cache'   # Colab si llega a Overpass; asi no se repite
!uv run city-analyser-models

## 2. Datos

Opcion A: copia a `MyDrive/city-analyser/data/` una carpeta de dataset de tu portatil (por ejemplo `data/valencia-centro`).

Opcion B: descargala con Mapillary (token en Colab > Secrets como `MAPILLARY_ACCESS_TOKEN`). Mira `uv run city-analyser-download --help` para el area.

In [ ]:
from google.colab import userdata
os.environ['MAPILLARY_ACCESS_TOKEN'] = userdata.get('MAPILLARY_ACCESS_TOKEN')
!uv run city-analyser-download --help | head -30

In [ ]:
DATASET = 'valencia-centro'   # cambia por tu dataset
!ls data/{DATASET} | head

## 3. Analisis completo en la GPU de Colab

## 3. Analisis de la ciudad: reanudable, con progreso visible

El analisis corre en segundo plano y puedes cerrarlo, reiniciarlo o perder la sesion: al volver a ejecutar esta celda continua donde
se quedo (los paneles terminados salen de `batch-ledger.jsonl` en menos de un segundo). Los errores transitorios se reintentan y los
paneles que fallan se reintentan en la siguiente ejecucion; un panel con datos malos se anota y no se repite.

In [ ]:
import subprocess
log = open(f'{WORK}/analyze-{DATASET}.log', 'a')
job = subprocess.Popen(['uv', 'run', 'city-analyser-analyze', DATASET, '--skip-low-quality', '--report', f'{WORK}/report-{DATASET}.json'],
                       stdout=log, stderr=subprocess.STDOUT, cwd='/content/repo')

Panel de progreso (barra, tiempo restante estimado, mapa de la ciudad por estados, ultimos errores). Se refresca solo y termina cuando acaba el analisis. Puedes interrumpir esta celda sin parar el analisis.

In [ ]:
import sys
sys.path.insert(0, '/content/repo/src')   # run_progress solo necesita Pillow, que ya trae Colab
from city_analyser.run_progress import watch
from pathlib import Path
watch(Path('/content/repo/data'), DATASET, every=15)


In [ ]:
!tail -5 {WORK}/analyze-{DATASET}.log
print('proceso:', 'en marcha' if job.poll() is None else f'terminado con codigo {job.returncode} (0 = ok, 1 = abortado por fallos seguidos, 130 = interrumpido)')

El informe tiene tiempos por panorama y memoria pico: son los numeros reales de GPU que el repo todavia no tiene.

Alternativa para repartir el trabajo: aqui solo `--perception-only` (llena la cache de modelos en Drive) y luego en tu portatil
`city-analyser-analyze` con la misma carpeta de cache (cada panorama tarda ~1 s).

## 4. Indice, recortes y atributos (OSM)

In [ ]:
!uv run city-analyser-index {DATASET} --select
!uv run city-analyser-crops {DATASET} --per-building 3 --mask-others
!uv run city-analyser-attributes {DATASET}
!uv run python -m evaluation.attributes --dataset {DATASET} --provider osm-tag-prior | head -40

In [ ]:
# Tabla por edificio (GeoJSON, CSV, GeoParquet si hay pyarrow) e informe de cobertura
!uv sync --extra cu126 --extra parquet
!uv run city-analyser-export {DATASET} --provider osm-tag-prior
!cat data/{DATASET}/exports/coverage.json | head -40


## 5. Atributos con un modelo de vision y lenguaje (opcional)

Se sirve con vLLM en un entorno aparte (no mezclar con el torch del proyecto) y el CLI lo llama por `localhost`.
Comprueba que el modelo existe y que vLLM lo soporta antes de lanzarlo; en una T4 hay que usar modelos pequenos (1B a 4B) en fp16.

In [ ]:
!pip -q install uv
!uv venv /content/vllm-env && uv pip install --python /content/vllm-env/bin/python vllm
import subprocess
MODEL = 'Qwen/Qwen3-VL-2B-Instruct'   # ejemplo: verifica el nombre exacto en Hugging Face
server = subprocess.Popen(['/content/vllm-env/bin/vllm', 'serve', MODEL, '--dtype', 'half', '--max-model-len', '4096', '--port', '8000'],
                          stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)

In [ ]:
!tail -5 /content/vllm.log   # espera a ver 'Application startup complete'

In [ ]:
!uv run city-analyser-attributes {DATASET} --provider vlm --vlm-url http://127.0.0.1:8000 --vlm-model {MODEL}
!uv run python -m evaluation.attributes --dataset {DATASET} --provider vlm_{MODEL.replace('/', '_')} | head -60

## 6. Resultados

Todo queda en Drive (`data/<dataset>/analysis`, `crops`, `attributes`, `building-index.json`). Descarga esas carpetas y
abre el visor en tu portatil con `make viewer`.